# MODULE 8-V2 — Compact DG-ConvoReleNet for Strict Subject-Independent MI-EEG

**Purpose:** corrected/revised version of Module 8 after the first pilot showed strong training-set fitting but weak cross-subject validation.

### What changed in V2
- compact Transformer: **2 layers / 4 heads / 128-d tokens**
- temporal kernels: **15 / 31 / 63**
- temporal patch stride: **16** → 40 Transformer tokens
- moderate dropout and stronger weight decay
- grouped **subject-level validation** with 25% of source subjects
- no target subject in normalization, validation, training, or model selection
- **GRL delayed to epoch 11** and capped at **0.05**
- domain-loss weight reduced to **0.02**
- Center Loss reduced to **0.005**
- validation model selection uses **balanced accuracy + macro-F1 only**; validation domain loss is never used for early stopping
- no LSTM
- optional diagnostic ablation: **A = CNN-Transformer, B = +Center, C = +Center+GRL**
- default proposed model is **C**, but the ablation pilot is marked engineering-only and must not be used to tune/report final test results under a strict protocol
- default execution is the **9-fold BCI-IV-2a pilot**, not the full 118-fold run

The frozen Module 6 interface remains unchanged: `(N,22,640)`, 160 Hz, 8–30 Hz, 4 s, 3 classes, source-only normalization and strict target isolation. The original 2026 transfer-learning paper motivates compact convolution + relational attention and conservative adaptation; its reported Tanh configuration achieved the strongest IV-2a result, while the recurrent extension did not improve the main result. 

In [1]:
# ============================================================
# CELL 1 — IMPORTS, REPRODUCIBILITY, DEVICE, PATH DISCOVERY
# ============================================================
from __future__ import annotations

import os
import json
import math
import time
import copy
import random
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict, replace
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

warnings.filterwarnings("ignore")

SEED = 20260824
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

# Robust project-root discovery for the existing Module 6 artifacts.
candidates = []
if os.environ.get("CROSS_DATASET_MI_PROJECT_ROOT"):
    candidates.append(Path(os.environ["CROSS_DATASET_MI_PROJECT_ROOT"]).expanduser())
candidates.extend([
    Path.home() / "Project2" / "cross_dataset_mi_project",
    Path.cwd() / "cross_dataset_mi_project",
    Path.home() / "cross_dataset_mi_project",
])

PROJECT_ROOT = next((p for p in candidates if p.exists()), candidates[0])
MANIFEST_ROOT = PROJECT_ROOT / "manifests"
CACHE_ROOT = PROJECT_ROOT / "cache"
RESULTS_ROOT = PROJECT_ROOT / "results"
MODULE8_ROOT = RESULTS_ROOT / "module_8_v2_dg_convorelenet"
CHECKPOINT_ROOT = MODULE8_ROOT / "checkpoints"
HISTORY_ROOT = MODULE8_ROOT / "histories"
PRED_ROOT = MODULE8_ROOT / "predictions"
PILOT_ROOT = MODULE8_ROOT / "pilot_ablation"
for p in [MODULE8_ROOT, CHECKPOINT_ROOT, HISTORY_ROOT, PRED_ROOT, PILOT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

CACHE_PATH = CACHE_ROOT / "module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5"
CACHE_META_PATH = MANIFEST_ROOT / "module_6_cache_metadata.csv"
WITHIN_LOSO_PATH = MANIFEST_ROOT / "module_6_within_dataset_loso_folds.csv"
TRANSFER_PATH = MANIFEST_ROOT / "module_6_cross_dataset_transfer_folds.csv"
PROTOCOL_PATH = MANIFEST_ROOT / "module_6_baseline_protocol.json"

REQUIRED = [CACHE_PATH, CACHE_META_PATH, WITHIN_LOSO_PATH, TRANSFER_PATH, PROTOCOL_PATH]

print("=" * 78)
print("MODULE 8-V2 — COMPACT DG-CONVORELENET")
print("=" * 78)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Device      :", DEVICE)
print("PyTorch     :", torch.__version__)
print("Seed        :", SEED)
for p in REQUIRED:
    print(("✓ " if p.exists() else "✗ ") + str(p))
assert all(p.exists() for p in REQUIRED), "Missing Module 6 artifacts."

MODULE 8-V2 — COMPACT DG-CONVORELENET
PROJECT_ROOT: /Users/ashokvarmabevara/Project2/cross_dataset_mi_project
Device      : mps
PyTorch     : 2.10.0
Seed        : 20260824
✓ /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/cache/module_5_v2_preprocessed_epochs_160hz_8_30hz_continuous.h5
✓ /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_6_cache_metadata.csv
✓ /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_6_within_dataset_loso_folds.csv
✓ /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_6_cross_dataset_transfer_folds.csv
✓ /Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_6_baseline_protocol.json


In [2]:
# ============================================================
# CELL 2 — FROZEN DATA SPECIFICATION + V2 CONTROLS
# ============================================================
PRIMARY_CLASSES = ["left", "right", "feet"]
CLASS_TO_ID = {"left": 0, "right": 1, "feet": 2}
ID_TO_CLASS = {v: k for k, v in CLASS_TO_ID.items()}
N_CHANNELS = 22
N_SAMPLES = 640
N_CLASSES = 3
TARGET_SFREQ = 160.0

@dataclass
class Config:
    # reproducibility
    seed: int = SEED

    # execution
    run_pilot_ablation: bool = True
    run_selected_within_full: bool = False
    run_cross_dataset: bool = False
    smoke_test: bool = False
    smoke_folds_per_dataset: int = 1
    selected_variant: str = "C"
    resume: bool = True
    save_predictions: bool = True
    plot_histories: bool = True

    # training
    batch_size: int = 64
    max_epochs: int = 40
    warmup_epochs: int = 10
    lr: float = 3e-4
    min_lr: float = 1e-5
    weight_decay: float = 3e-4
    patience: int = 7
    grad_clip: float = 1.0
    label_smoothing: float = 0.05

    # validation / model selection
    validation_subject_fraction: float = 0.25
    validation_seed_offset: int = 901

    # loss weights
    center_loss_weight: float = 0.005
    domain_loss_weight: float = 0.02
    grl_max_lambda: float = 0.05
    center_lr: float = 0.05

    # architecture
    temporal_channels_each: int = 16
    fusion_channels: int = 48
    spatial_channels: int = 64
    token_dim: int = 128
    patch_stride: int = 16
    transformer_layers: int = 2
    transformer_heads: int = 4
    transformer_ffn: int = 256
    transformer_dropout: float = 0.20
    embedding_dim: int = 128
    classifier_dropout: float = 0.30
    use_lstm: bool = False

    # training-only EEG augmentation
    train_noise_std: float = 0.005
    train_amp_jitter: float = 0.03
    train_channel_dropout: float = 0.03
    train_time_mask_prob: float = 0.10
    train_time_mask_max: int = 24

CFG = Config()
print(json.dumps(asdict(CFG), indent=2))


{
  "seed": 20260824,
  "run_pilot_ablation": true,
  "run_selected_within_full": false,
  "run_cross_dataset": false,
  "smoke_test": false,
  "smoke_folds_per_dataset": 1,
  "selected_variant": "C",
  "resume": true,
  "save_predictions": true,
  "plot_histories": true,
  "batch_size": 64,
  "max_epochs": 40,
  "warmup_epochs": 10,
  "lr": 0.0003,
  "min_lr": 1e-05,
  "weight_decay": 0.0003,
  "patience": 7,
  "grad_clip": 1.0,
  "label_smoothing": 0.05,
  "validation_subject_fraction": 0.25,
  "validation_seed_offset": 901,
  "center_loss_weight": 0.005,
  "domain_loss_weight": 0.02,
  "grl_max_lambda": 0.05,
  "center_lr": 0.05,
  "temporal_channels_each": 16,
  "fusion_channels": 48,
  "spatial_channels": 64,
  "token_dim": 128,
  "patch_stride": 16,
  "transformer_layers": 2,
  "transformer_heads": 4,
  "transformer_ffn": 256,
  "transformer_dropout": 0.2,
  "embedding_dim": 128,
  "classifier_dropout": 0.3,
  "use_lstm": false,
  "train_noise_std": 0.005,
  "train_amp_jitter": 0

In [3]:
# ============================================================
# CELL 3 — LOAD FROZEN METADATA + MANIFESTS
# ============================================================
cache_meta_df = pd.read_csv(CACHE_META_PATH)
within_loso_df = pd.read_csv(WITHIN_LOSO_PATH)
transfer_df = pd.read_csv(TRANSFER_PATH)

assert len(cache_meta_df) == 9316, len(cache_meta_df)
assert cache_meta_df["subject"].nunique() == 118
assert set(cache_meta_df["harmonized_class"].unique()) == set(PRIMARY_CLASSES)
assert len(within_loso_df) == 118
assert len(transfer_df) == 118

print("Cache epochs      :", len(cache_meta_df))
print("Unique subjects   :", cache_meta_df["subject"].nunique())
print("Within LOSO folds :", len(within_loso_df))
print("Transfer folds    :", len(transfer_df))
print("Class counts:")
print(cache_meta_df["harmonized_class"].value_counts().sort_index())


Cache epochs      : 9316
Unique subjects   : 118
Within LOSO folds : 118
Transfer folds    : 118
Class counts:
harmonized_class
feet     3103
left     3127
right    3086
Name: count, dtype: int64


In [4]:
# ============================================================
# CELL 4 — HDF5 STORE + SOURCE-ONLY ROBUST NORMALIZER
# ============================================================
class HDF5Store:
    def __init__(self, path):
        self.path = Path(path)
        self.h5 = None
    def __enter__(self):
        self.h5 = h5py.File(self.path, "r")
        return self
    def __exit__(self, exc_type, exc, tb):
        if self.h5 is not None:
            self.h5.close()
        self.h5 = None
    def get_X(self, indices):
        indices = np.asarray(indices, dtype=np.int64)
        if len(indices) == 0:
            return np.empty((0, N_CHANNELS, N_SAMPLES), dtype=np.float32)
        order = np.argsort(indices)
        sorted_idx = indices[order]
        X_sorted = self.h5["X"][sorted_idx]
        inverse = np.argsort(order)
        return np.asarray(X_sorted[inverse], dtype=np.float32)

class SourceOnlyRobustNormalizer:
    def __init__(self, eps=1e-6):
        self.eps = float(eps)
        self.median_ = None
        self.iqr_ = None
        self.fitted_subjects_ = tuple()
    def fit(self, X_source, source_subjects):
        X_source = np.asarray(X_source, dtype=np.float64)
        source_subjects = [str(s) for s in source_subjects]
        assert X_source.shape[1:] == (N_CHANNELS, N_SAMPLES)
        assert len(source_subjects) == len(X_source)
        values = X_source.transpose(1, 0, 2).reshape(N_CHANNELS, -1)
        self.median_ = np.median(values, axis=1)
        q25 = np.percentile(values, 25, axis=1)
        q75 = np.percentile(values, 75, axis=1)
        self.iqr_ = np.maximum(q75 - q25, self.eps)
        self.fitted_subjects_ = tuple(sorted(set(source_subjects)))
        return self
    def transform(self, X):
        assert self.median_ is not None
        X = np.asarray(X, dtype=np.float32)
        return ((X - self.median_[None, :, None]) / self.iqr_[None, :, None]).astype(np.float32)
    def assert_target_excluded(self, target_subject):
        if str(target_subject) in self.fitted_subjects_:
            raise AssertionError(f"Normalization leakage: {target_subject}")

def compute_class_ids(meta):
    return np.asarray([CLASS_TO_ID[x] for x in meta["harmonized_class"]], dtype=np.int64)

def safe_indices(json_string):
    return np.asarray(json.loads(json_string), dtype=np.int64)


In [5]:
# ============================================================
# CELL 5 — STRICT GROUPED SOURCE TRAIN/VALIDATION SPLIT
# ============================================================
def grouped_source_split(source_indices, meta, target_subject=None, seed=0, val_fraction=0.25):
    source_indices = np.asarray(source_indices, dtype=np.int64)
    source_meta = meta.loc[source_indices].copy()
    if target_subject is not None:
        assert str(target_subject) not in set(source_meta["subject"].astype(str))
    subjects = np.array(sorted(source_meta["subject"].astype(str).unique()))
    assert len(subjects) >= 3, "Need at least 3 source subjects for grouped train/validation."
    rng = np.random.default_rng(seed)
    shuffled = subjects.copy()
    rng.shuffle(shuffled)
    n_val = max(2, int(round(len(shuffled) * val_fraction)))
    n_val = min(n_val, len(shuffled) - 1)
    val_subjects = set(shuffled[:n_val])
    val_mask = source_meta["subject"].astype(str).isin(val_subjects).to_numpy()
    train_idx = source_indices[~val_mask]
    val_idx = source_indices[val_mask]
    assert len(train_idx) > 0 and len(val_idx) > 0
    train_subjects = set(meta.loc[train_idx, "subject"].astype(str))
    eval_subjects = set(meta.loc[val_idx, "subject"].astype(str))
    assert train_subjects.isdisjoint(eval_subjects)
    if target_subject is not None:
        assert str(target_subject) not in train_subjects
        assert str(target_subject) not in eval_subjects
    return train_idx, val_idx, sorted(val_subjects)


In [6]:
# ============================================================
# CELL 6 — TRAINING-ONLY EEG AUGMENTATION + DATASETS
# ============================================================
class EEGAugment:
    def __init__(self, noise_std, amp_jitter, channel_dropout, time_mask_prob, time_mask_max):
        self.noise_std = float(noise_std)
        self.amp_jitter = float(amp_jitter)
        self.channel_dropout = float(channel_dropout)
        self.time_mask_prob = float(time_mask_prob)
        self.time_mask_max = int(time_mask_max)
    def __call__(self, x):
        x = x.clone()
        if self.noise_std > 0:
            x = x + torch.randn_like(x) * self.noise_std
        if self.amp_jitter > 0:
            scale = 1.0 + (2 * torch.rand((x.shape[0], 1)) - 1.0) * self.amp_jitter
            x = x * scale
        if self.channel_dropout > 0 and torch.rand(()) < self.channel_dropout:
            c = int(torch.randint(0, x.shape[0], (1,)).item())
            x[c] = 0.0
        if self.time_mask_prob > 0 and torch.rand(()) < self.time_mask_prob:
            width = int(torch.randint(8, self.time_mask_max + 1, (1,)).item())
            width = min(width, x.shape[1])
            start = int(torch.randint(0, max(1, x.shape[1] - width + 1), (1,)).item())
            x[:, start:start+width] = 0.0
        return x

class ArrayEEGDataset(Dataset):
    def __init__(self, X, y, subjects, augment=None):
        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.int64)
        self.subjects = np.asarray(subjects, dtype=np.int64)
        self.augment = augment
        assert self.X.shape[1:] == (N_CHANNELS, N_SAMPLES)
        assert len(self.X) == len(self.y) == len(self.subjects)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx])
        if self.augment is not None:
            x = self.augment(x)
        return x, torch.tensor(self.y[idx], dtype=torch.long), torch.tensor(self.subjects[idx], dtype=torch.long)

class TensorEEGOnlyDataset(Dataset):
    def __init__(self, X):
        self.X = np.asarray(X, dtype=np.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return torch.from_numpy(self.X[i])


In [7]:
# ============================================================
# CELL 7 — GRL, CENTER LOSS, ATTENTION POOLING
# ============================================================
class GradientReversalFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = float(lambd)
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None

def grad_reverse(x, lambd):
    return GradientReversalFn.apply(x, lambd)

class CenterLoss(nn.Module):
    def __init__(self, num_classes, feat_dim):
        super().__init__()
        self.centers = nn.Parameter(torch.randn(num_classes, feat_dim) * 0.02)
    def forward(self, features, labels):
        centers_batch = self.centers.index_select(0, labels)
        return 0.5 * ((features - centers_batch) ** 2).sum(dim=1).mean()

class AttentionPooling(nn.Module):
    def __init__(self, dim):
        super().__init__()
        hidden = max(16, dim // 2)
        self.score = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))
    def forward(self, tokens):
        weights = torch.softmax(self.score(tokens).squeeze(-1), dim=1)
        pooled = torch.sum(tokens * weights.unsqueeze(-1), dim=1)
        return pooled, weights

def variant_settings(variant, base_cfg):
    variant = str(variant).upper()
    assert variant in {"A", "B", "C"}
    c = replace(base_cfg)
    if variant == "A":
        c.center_loss_weight = 0.0
        c.domain_loss_weight = 0.0
        c.grl_max_lambda = 0.0
    elif variant == "B":
        c.center_loss_weight = base_cfg.center_loss_weight
        c.domain_loss_weight = 0.0
        c.grl_max_lambda = 0.0
    else:
        c.center_loss_weight = base_cfg.center_loss_weight
        c.domain_loss_weight = base_cfg.domain_loss_weight
        c.grl_max_lambda = base_cfg.grl_max_lambda
    return c


In [8]:
# ============================================================
# CELL 8 — COMPACT DG-CONVORELENET V2
# ============================================================
class TemporalBranch(nn.Module):
    def __init__(self, out_channels, kernel_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, out_channels, kernel_size=(1, kernel_size), padding=(0, kernel_size // 2), bias=False),
            nn.BatchNorm2d(out_channels),
            nn.Tanh(),
            nn.Dropout2d(0.08),
        )
    def forward(self, x): return self.net(x)

class DGConvoReleNetV2(nn.Module):
    def __init__(self, num_classes=3, num_domains=2, cfg=CFG, use_domain=True):
        super().__init__()
        self.use_domain = bool(use_domain and cfg.domain_loss_weight > 0 and cfg.grl_max_lambda > 0)
        kernels = [15, 31, 63]
        self.temporal = nn.ModuleList([TemporalBranch(cfg.temporal_channels_each, k) for k in kernels])
        self.spatial = nn.Sequential(
            nn.Conv2d(cfg.fusion_channels, cfg.spatial_channels, kernel_size=(N_CHANNELS, 1), bias=False),
            nn.BatchNorm2d(cfg.spatial_channels),
            nn.Tanh(),
            nn.Dropout2d(0.10),
        )
        self.patch = nn.Sequential(
            nn.Conv2d(cfg.spatial_channels, cfg.token_dim, kernel_size=(1, cfg.patch_stride), stride=(1, cfg.patch_stride), bias=False),
            nn.BatchNorm2d(cfg.token_dim),
            nn.Tanh(),
        )
        max_tokens = N_SAMPLES // cfg.patch_stride
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=cfg.token_dim,
            nhead=cfg.transformer_heads,
            dim_feedforward=cfg.transformer_ffn,
            dropout=cfg.transformer_dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=cfg.transformer_layers)
        self.positional = nn.Parameter(torch.zeros(1, max_tokens, cfg.token_dim))
        nn.init.normal_(self.positional, std=0.01)
        self.attn_pool = AttentionPooling(cfg.token_dim)
        self.embedding = nn.Sequential(
            nn.Linear(cfg.token_dim, cfg.embedding_dim),
            nn.Tanh(),
            nn.LayerNorm(cfg.embedding_dim),
            nn.Dropout(cfg.classifier_dropout),
        )
        self.classifier = nn.Linear(cfg.embedding_dim, num_classes)
        if self.use_domain:
            self.domain_classifier = nn.Sequential(
                nn.Linear(cfg.embedding_dim, 64),
                nn.Tanh(),
                nn.Dropout(0.20),
                nn.Linear(64, num_domains),
            )
        else:
            self.domain_classifier = None

    def forward(self, x, grl_lambda=0.0):
        x = x.unsqueeze(1)
        x = torch.cat([b(x) for b in self.temporal], dim=1)
        x = self.spatial(x)
        x = self.patch(x).squeeze(2).transpose(1, 2)
        L = x.shape[1]
        x = x + self.positional[:, :L]
        x = self.transformer(x)
        pooled, attn = self.attn_pool(x)
        emb = self.embedding(pooled)
        logits = self.classifier(emb)
        domain_logits = None
        if self.use_domain:
            domain_logits = self.domain_classifier(grad_reverse(emb, grl_lambda))
        return {"logits": logits, "embedding": emb, "attention": attn, "domain_logits": domain_logits}

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

for v in ["A", "B", "C"]:
    cv = variant_settings(v, CFG)
    m = DGConvoReleNetV2(3, num_domains=8, cfg=cv, use_domain=(v == "C")).to(DEVICE)
    with torch.no_grad():
        o = m(torch.randn(2, 22, 640, device=DEVICE))
    print(v, "params=", f"{count_parameters(m):,}", "logits=", tuple(o["logits"].shape), "emb=", tuple(o["embedding"].shape))
    del m, o


A params= 496,436 logits= (2, 3) emb= (2, 128)
B params= 496,436 logits= (2, 3) emb= (2, 128)
C params= 505,212 logits= (2, 3) emb= (2, 128)


In [9]:
# ============================================================
# CELL 9 — BALANCED SUBJECT+CLASS SAMPLING + SEED + METRICS
# ============================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def make_loaders(X_train, y_train, s_train, X_val, y_val, s_val, cfg):
    aug = EEGAugment(cfg.train_noise_std, cfg.train_amp_jitter, cfg.train_channel_dropout, cfg.train_time_mask_prob, cfg.train_time_mask_max)
    train_ds = ArrayEEGDataset(X_train, y_train, s_train, augment=aug)
    val_ds = ArrayEEGDataset(X_val, y_val, s_val, augment=None)

    # Balance both source subjects and classes.
    # This prevents high-trial subjects/classes from dominating training.
    pairs = pd.DataFrame({"s": np.asarray(s_train), "y": np.asarray(y_train)})
    pair_counts = pairs.groupby(["s", "y"]).size().to_dict()
    subject_counts = pairs.groupby("s").size().to_dict()
    weights = []
    for s, y in zip(s_train, y_train):
        w_subject = 1.0 / max(1, subject_counts[int(s)])
        w_class = 1.0 / max(1, pair_counts[(int(s), int(y))])
        weights.append(w_subject * w_class)
    weights = np.asarray(weights, dtype=np.float64)
    weights = weights / weights.mean()
    sampler = WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), num_samples=len(weights), replacement=True)
    pin = DEVICE.type == "cuda"
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, sampler=sampler, shuffle=False, num_workers=0, pin_memory=pin, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size * 2, shuffle=False, num_workers=0, pin_memory=pin)
    return train_loader, val_loader

def compute_all_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "confusion_matrix": json.dumps(cm.tolist()),
    }

def scheduled_grl_lambda(epoch, cfg):
    if cfg.grl_max_lambda <= 0 or epoch <= cfg.warmup_epochs:
        return 0.0
    p = (epoch - cfg.warmup_epochs) / max(1, cfg.max_epochs - cfg.warmup_epochs)
    p = min(max(p, 0.0), 1.0)
    # Smooth ramp starting from 0 after warm-up and approaching max_lambda.
    return float(cfg.grl_max_lambda * (2.0 / (1.0 + math.exp(-8.0 * (p - 0.5))) - 1.0))


In [10]:
# ============================================================
# CELL 10 — ONE-FOLD TRAINER V2
# ============================================================
def train_one_fold_v2(X_train, y_train, subj_train, X_val, y_val, subj_val, cfg, variant, seed, fold_tag, num_domains=None):
    set_seed(seed)
    variant = str(variant).upper()
    use_domain = variant == "C"

    # Map source training subjects to contiguous domain IDs. Validation subjects
    # are never passed to the domain head because they are unseen domains.
    unique_train_subjects = sorted(set(map(int, subj_train)))
    subj_to_domain = {s: i for i, s in enumerate(unique_train_subjects)}
    domain_train = np.asarray([subj_to_domain[int(s)] for s in subj_train], dtype=np.int64)
    domain_count = max(2, len(unique_train_subjects)) if use_domain else 2

    train_loader, val_loader = make_loaders(X_train, y_train, subj_train, X_val, y_val, subj_val, cfg)
    model = DGConvoReleNetV2(N_CLASSES, domain_count, cfg=cfg, use_domain=use_domain).to(DEVICE)
    center = CenterLoss(N_CLASSES, cfg.embedding_dim).to(DEVICE)

    model_optim = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    center_optim = torch.optim.SGD(center.parameters(), lr=cfg.center_lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(model_optim, T_max=cfg.max_epochs, eta_min=cfg.min_lr)

    ce = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    dom_ce = nn.CrossEntropyLoss()
    best_state = None
    best_center = None
    best_score = -np.inf
    wait = 0
    history = []

    for epoch in range(1, cfg.max_epochs + 1):
        model.train(); center.train()
        grl = scheduled_grl_lambda(epoch, cfg) if use_domain else 0.0
        train_sum = 0.0; train_n = 0; train_correct = 0

        for xb, yb, db in train_loader:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE); db = db.to(DEVICE)
            model_optim.zero_grad(set_to_none=True); center_optim.zero_grad(set_to_none=True)
            out = model(xb, grl_lambda=grl)
            cls_loss = ce(out["logits"], yb)
            c_loss = center(out["embedding"], yb) if variant in {"B", "C"} else torch.zeros((), device=DEVICE)
            d_loss = dom_ce(out["domain_logits"], db) if use_domain else torch.zeros((), device=DEVICE)
            loss = cls_loss + cfg.center_loss_weight * c_loss + cfg.domain_loss_weight * d_loss
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            nn.utils.clip_grad_norm_(center.parameters(), 1.0)
            model_optim.step()
            if variant in {"B", "C"}:
                # Lower center step frequency prevents center drift under strong stochastic augmentation.
                center_optim.step()
            train_sum += float(loss.detach().cpu()) * len(xb)
            train_n += len(xb)
            train_correct += int((out["logits"].argmax(1) == yb).sum().detach().cpu())

        scheduler.step()

        # Validation: only classification and embedding center loss are relevant.
        # No domain loss is evaluated because validation subjects are unseen domains.
        model.eval(); center.eval()
        y_true=[]; y_pred=[]; val_sum=0.0
        with torch.no_grad():
            for xb, yb, _ in val_loader:
                xb=xb.to(DEVICE); yb=yb.to(DEVICE)
                out=model(xb, grl_lambda=0.0)
                cls_loss=ce(out["logits"], yb)
                c_loss=center(out["embedding"], yb) if variant in {"B","C"} else torch.zeros((), device=DEVICE)
                val_loss=cls_loss + cfg.center_loss_weight*c_loss
                val_sum += float(val_loss.detach().cpu()) * len(xb)
                y_true.extend(yb.cpu().numpy().tolist())
                y_pred.extend(out["logits"].argmax(1).cpu().numpy().tolist())

        val_metrics=compute_all_metrics(np.asarray(y_true), np.asarray(y_pred))
        train_acc=train_correct/max(1,train_n)
        train_loss=train_sum/max(1,train_n)
        val_loss=val_sum/max(1,len(y_true))
        score=val_metrics["balanced_accuracy"] + 0.20*val_metrics["macro_f1"]
        row={"epoch":epoch,"train_loss":train_loss,"train_accuracy":train_acc,"val_loss":val_loss,"val_accuracy":val_metrics["accuracy"],"val_balanced_accuracy":val_metrics["balanced_accuracy"],"val_macro_f1":val_metrics["macro_f1"],"grl_lambda":grl,"lr":model_optim.param_groups[0]["lr"]}
        history.append(row)

        if score > best_score + 1e-5:
            best_score=score
            best_state=copy.deepcopy(model.state_dict())
            best_center=copy.deepcopy(center.state_dict())
            wait=0
        else:
            wait += 1

        print(f"{fold_tag} | ep {epoch:02d}/{cfg.max_epochs} | train {train_acc:.3f} | val {val_metrics['accuracy']:.3f} bal {val_metrics['balanced_accuracy']:.3f} | loss {val_loss:.4f} | GRL {grl:.3f}")
        if wait >= cfg.patience:
            print(f"{fold_tag} | early stopping at epoch {epoch}")
            break

    assert best_state is not None
    model.load_state_dict(best_state); center.load_state_dict(best_center)
    return model, center, pd.DataFrame(history)


In [11]:
# ============================================================
# CELL 11 — INFERENCE + CHECKPOINTS
# ============================================================
def predict_model(model, X, batch_size=256):
    model.eval()
    loader = DataLoader(TensorEEGOnlyDataset(X), batch_size=batch_size, shuffle=False, num_workers=0)
    preds=[]; probs=[]; embeddings=[]
    with torch.no_grad():
        for xb in loader:
            xb=xb.to(DEVICE)
            out=model(xb, grl_lambda=0.0)
            p=torch.softmax(out["logits"], dim=1)
            preds.append(p.argmax(1).cpu().numpy())
            probs.append(p.cpu().numpy())
            embeddings.append(out["embedding"].cpu().numpy())
    return np.concatenate(preds), np.concatenate(probs), np.concatenate(embeddings)

def save_checkpoint(path, model, center, cfg, metadata):
    torch.save({"model_state":model.state_dict(),"center_state":center.state_dict(),"config":asdict(cfg),"metadata":metadata}, path)


## CELL 12 — Strict protocol and variant interpretation

**A — Compact CNN-Transformer:** architecture-only baseline.

**B — + Center Loss:** tests whether tighter intra-class embeddings improve cross-subject decoding.

**C — + Center Loss + delayed weak subject-adversarial GRL:** proposed V2 model.

For every strict LOSO fold:

1. target subject is removed before normalization and fitting;
2. source subjects are split into grouped train/validation subjects;
3. normalization is fit on train subjects only;
4. validation subjects are not used in domain-head supervision;
5. final target evaluation occurs once, after training is finished.

The A/B/C pilot is **engineering diagnostics only**. Do not use the held-out target results from the pilot to tune a model and then claim those same target results as an untouched final evaluation. For the publication-facing run, lock the selected configuration first and then run the full LOSO protocol exactly once.

In [12]:
# ============================================================
# CELL 13 — ONE WITHIN-DATASET LOSO FOLD
# ============================================================
def run_within_fold_v2(fold_row, variant="C", save_outputs=True):
    dataset=str(fold_row["dataset"]); target_subject=str(fold_row["target_subject"]); fold_id=int(fold_row["fold_id"])
    source_indices=safe_indices(fold_row["train_indices_json"]); test_indices=safe_indices(fold_row["test_indices_json"])
    train_indices,val_indices,val_subjects=grouped_source_split(source_indices,cache_meta_df,target_subject=target_subject,seed=CFG.seed+CFG.validation_seed_offset+fold_id,val_fraction=CFG.validation_subject_fraction)

    with HDF5Store(CACHE_PATH) as store:
        X_train_raw=store.get_X(train_indices); X_val_raw=store.get_X(val_indices); X_test_raw=store.get_X(test_indices)
    train_meta=cache_meta_df.loc[train_indices]; val_meta=cache_meta_df.loc[val_indices]; test_meta=cache_meta_df.loc[test_indices]
    normalizer=SourceOnlyRobustNormalizer().fit(X_train_raw,train_meta["subject"].astype(str).tolist())
    normalizer.assert_target_excluded(target_subject)
    X_train=normalizer.transform(X_train_raw); X_val=normalizer.transform(X_val_raw); X_test=normalizer.transform(X_test_raw)
    y_train=compute_class_ids(train_meta); y_val=compute_class_ids(val_meta); y_test=compute_class_ids(test_meta)

    source_subjects=sorted(train_meta["subject"].astype(str).unique())
    subject_to_int={s:i for i,s in enumerate(source_subjects)}
    s_train=np.asarray([subject_to_int[str(s)] for s in train_meta["subject"]],dtype=np.int64)
    # Validation labels are irrelevant to domain training and remain separate.
    s_val=np.zeros(len(val_meta),dtype=np.int64)

    assert set(test_meta["subject"].astype(str))=={target_subject}
    assert set(train_meta["subject"].astype(str)).isdisjoint(set(val_meta["subject"].astype(str)))

    local_cfg=variant_settings(variant,CFG)
    start=time.time()
    model,center,history=train_one_fold_v2(X_train,y_train,s_train,X_val,y_val,s_val,local_cfg,variant,CFG.seed+fold_id,f"{variant} | WITHIN {dataset}/{target_subject}")
    preds,probs,embeddings=predict_model(model,X_test)
    metrics=compute_all_metrics(y_test,preds)

    row={**metrics,"variant":variant,"protocol":"within_dataset_loso","dataset":dataset,"target_subject":target_subject,"fold_id":fold_id,"source_train_epochs":len(train_indices),"source_val_epochs":len(val_indices),"val_subjects":json.dumps(val_subjects),"epochs_run":len(history),"runtime_sec":time.time()-start,"best_val_balanced_accuracy":float(history["val_balanced_accuracy"].max()),"best_val_accuracy":float(history["val_accuracy"].max()),"best_val_macro_f1":float(history["val_macro_f1"].max())}

    if save_outputs:
        prefix=f"pilot_{variant}_{dataset}_{target_subject}" if CFG.run_pilot_ablation else f"selected_{variant}_{dataset}_{target_subject}"
        save_checkpoint(CHECKPOINT_ROOT/f"{prefix}.pt",model,center,local_cfg,{"variant":variant,"protocol":"within_dataset_loso","dataset":dataset,"target_subject":target_subject,"fold_id":fold_id,"val_subjects":val_subjects})
        history.to_csv(HISTORY_ROOT/f"{prefix}.csv",index=False)
        if CFG.save_predictions:
            pred_df=test_meta[["cache_index","dataset","subject","harmonized_class"]].copy()
            pred_df["true_id"]=y_test; pred_df["pred_id"]=preds; pred_df["pred_class"]= [ID_TO_CLASS[int(x)] for x in preds]
            for k in range(N_CLASSES): pred_df[f"prob_{ID_TO_CLASS[k]}"]=probs[:,k]
            pred_df.to_csv(PRED_ROOT/f"{prefix}.csv",index=False)
    print("="*78)
    print(f"DONE | {variant} | {dataset} | {target_subject} | accuracy={metrics['accuracy']:.4f} | balanced={metrics['balanced_accuracy']:.4f} | F1={metrics['macro_f1']:.4f}")
    print("Runtime sec:",round(time.time()-start,2))
    return row


In [ ]:
# ============================================================
# CELL 14 — ENGINEERING PILOT: BCI-IV-2a S01-S09, A/B/C
# ============================================================
PILOT_RESULTS_PATH=PILOT_ROOT/"v2_bci2a_ablation_results.csv"

def run_pilot_ablation():
    folds=within_loso_df[within_loso_df["dataset"].astype(str)=="BCI-IV-2a"].copy()
    if CFG.smoke_test:
        folds=folds.head(CFG.smoke_folds_per_dataset).copy()
    variants=["A","B","C"]

    existing=pd.read_csv(PILOT_RESULTS_PATH) if PILOT_RESULTS_PATH.exists() and CFG.resume else pd.DataFrame()
    done=set() if len(existing)==0 else set(existing["variant"].astype(str)+"::"+existing["target_subject"].astype(str))
    rows=[]
    for variant in variants:
        for _,fold in folds.iterrows():
            key=f"{variant}::{fold['target_subject']}"
            if key in done:
                print("SKIP existing:",key)
                continue
            row=run_within_fold_v2(fold,variant=variant,save_outputs=True)
            rows.append(row)
            existing=pd.concat([existing,pd.DataFrame([row])],ignore_index=True)
            existing.to_csv(PILOT_RESULTS_PATH,index=False)
    return pd.read_csv(PILOT_RESULTS_PATH) if PILOT_RESULTS_PATH.exists() else pd.DataFrame(rows)

pilot_results=pd.DataFrame()
if CFG.run_pilot_ablation:
    pilot_results=run_pilot_ablation()
    print("Saved:",PILOT_RESULTS_PATH)
else:
    print("Pilot ablation disabled.")


A | WITHIN BCI-IV-2a/S01 | ep 01/40 | train 0.366 | val 0.329 bal 0.329 | loss 1.1293 | GRL 0.000
A | WITHIN BCI-IV-2a/S01 | ep 02/40 | train 0.404 | val 0.343 bal 0.343 | loss 1.1071 | GRL 0.000
A | WITHIN BCI-IV-2a/S01 | ep 03/40 | train 0.439 | val 0.396 bal 0.396 | loss 1.0778 | GRL 0.000
A | WITHIN BCI-IV-2a/S01 | ep 04/40 | train 0.531 | val 0.463 bal 0.463 | loss 1.1213 | GRL 0.000
A | WITHIN BCI-IV-2a/S01 | ep 05/40 | train 0.535 | val 0.396 bal 0.396 | loss 1.0722 | GRL 0.000
A | WITHIN BCI-IV-2a/S01 | ep 06/40 | train 0.593 | val 0.433 bal 0.433 | loss 1.1701 | GRL 0.000
A | WITHIN BCI-IV-2a/S01 | ep 07/40 | train 0.628 | val 0.428 bal 0.428 | loss 1.1734 | GRL 0.000
A | WITHIN BCI-IV-2a/S01 | ep 08/40 | train 0.688 | val 0.442 bal 0.442 | loss 1.2035 | GRL 0.000
A | WITHIN BCI-IV-2a/S01 | ep 09/40 | train 0.718 | val 0.461 bal 0.461 | loss 1.3579 | GRL 0.000
A | WITHIN BCI-IV-2a/S01 | ep 10/40 | train 0.763 | val 0.417 bal 0.417 | loss 1.4305 | GRL 0.000
A | WITHIN BCI-IV-2a

In [ ]:
# ============================================================
# CELL 15 — PILOT SUMMARY + RECOMMENDED VARIANT
# ============================================================
def summarize_variant_results(df):
    if df is None or len(df)==0:
        return pd.DataFrame()
    out=df.groupby("variant").agg(
        accuracy_mean=("accuracy","mean"),accuracy_std=("accuracy","std"),
        balanced_mean=("balanced_accuracy","mean"),balanced_std=("balanced_accuracy","std"),
        f1_mean=("macro_f1","mean"),f1_std=("macro_f1","std"),
        mean_best_val=("best_val_balanced_accuracy","mean"),
        target_count=("target_subject","count")
    ).reset_index()
    return out.sort_values(["balanced_mean","f1_mean"],ascending=False)

if len(pilot_results):
    summary=summarize_variant_results(pilot_results)
    display(summary.assign(
        accuracy_mean_pct=100*summary.accuracy_mean,
        balanced_mean_pct=100*summary.balanced_mean,
        f1_mean_pct=100*summary.f1_mean,
    )[["variant","accuracy_mean_pct","accuracy_std","balanced_mean_pct","balanced_std","f1_mean_pct","f1_std","mean_best_val","target_count"]])
    print("\nWARNING: this is an engineering pilot; do NOT use these held-out target results to tune and then report them as final strict LOSO results.")


In [ ]:
# ============================================================
# CELL 16 — SELECTED V2 FULL WITHIN-LOSO RUNNER (OPT-IN)
# ============================================================
SELECTED_RESULT_PATH=MODULE8_ROOT/"selected_v2_within_loso_results.csv"

def run_selected_within_full():
    folds=within_loso_df.copy()
    if CFG.smoke_test:
        folds=folds.groupby("dataset",group_keys=False).head(CFG.smoke_folds_per_dataset).copy()
    existing=pd.read_csv(SELECTED_RESULT_PATH) if SELECTED_RESULT_PATH.exists() and CFG.resume else pd.DataFrame()
    done=set() if len(existing)==0 else set(existing["dataset"].astype(str)+"::"+existing["target_subject"].astype(str))
    for _,fold in folds.iterrows():
        key=f"{fold['dataset']}::{fold['target_subject']}"
        if key in done:
            print("SKIP existing:",key); continue
        row=run_within_fold_v2(fold,variant=CFG.selected_variant,save_outputs=True)
        existing=pd.concat([existing,pd.DataFrame([row])],ignore_index=True)
        existing.to_csv(SELECTED_RESULT_PATH,index=False)
    return pd.read_csv(SELECTED_RESULT_PATH) if SELECTED_RESULT_PATH.exists() else pd.DataFrame()

selected_within_results=pd.DataFrame()
if CFG.run_selected_within_full:
    selected_within_results=run_selected_within_full()
else:
    print("Selected full LOSO disabled. Set CFG.run_selected_within_full=True after the V2 configuration is locked.")


In [ ]:
# ============================================================
# CELL 17 — OPTIONAL CROSS-DATASET ZERO-CALIBRATION SKELETON
# ============================================================
# The cross-dataset run is deliberately disabled in V2 until the within-dataset
# configuration is locked. Reuse the exact source-only normalization and strict
# target-isolation logic from Module 6.
if CFG.run_cross_dataset:
    raise NotImplementedError(
        "V2 cross-dataset execution is intentionally locked until the within-dataset model is selected. "
        "Set run_cross_dataset=False for the current pilot."
    )
else:
    print("Cross-dataset zero-calibration is locked OFF in the V2 pilot.")


In [ ]:
# ============================================================
# CELL 18 — HISTORY PLOTS + PER-SUBJECT DIAGNOSTICS
# ============================================================
if CFG.plot_histories:
    hist_files=sorted(HISTORY_ROOT.glob("pilot_*.csv"))
    for hp in hist_files[:min(6,len(hist_files))]:
        h=pd.read_csv(hp)
        fig=plt.figure(figsize=(8,4))
        plt.plot(h["epoch"],h["train_accuracy"],label="Train")
        plt.plot(h["epoch"],h["val_balanced_accuracy"],label="Validation balanced")
        plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title(hp.stem); plt.legend(); plt.tight_layout(); plt.show()

if len(pilot_results):
    by_subject=pilot_results.pivot_table(index="target_subject",columns="variant",values="accuracy")
    print("Pilot accuracy by subject:")
    display(100*by_subject)


In [ ]:
# ============================================================
# CELL 19 — MODEL / PROTOCOL SPECIFICATION + FINAL QA
# ============================================================
SPEC_OUTPUT=MODULE8_ROOT/"module_8_v2_model_and_protocol_specification.json"
spec={
    "module":"8-V2",
    "model":"Compact DG-ConvoReleNet",
    "input_shape":[22,640],
    "sampling_rate_hz":160.0,
    "band_hz":[8.0,30.0],
    "classes":PRIMARY_CLASSES,
    "epochs_total":int(len(cache_meta_df)),
    "subjects_total":int(cache_meta_df["subject"].nunique()),
    "architecture":{
        "temporal_kernels":[15,31,63],
        "temporal_channels_each":CFG.temporal_channels_each,
        "spatial_channels":CFG.spatial_channels,
        "token_dim":CFG.token_dim,
        "patch_stride":CFG.patch_stride,
        "transformer_layers":CFG.transformer_layers,
        "transformer_heads":CFG.transformer_heads,
        "transformer_ffn":CFG.transformer_ffn,
        "embedding_dim":CFG.embedding_dim,
        "activation":"Tanh",
        "attention_pooling":True,
        "lstm":False,
    },
    "v2_changes":{
        "grouped_validation_fraction":CFG.validation_subject_fraction,
        "grl_warmup_epochs":CFG.warmup_epochs,
        "grl_max_lambda":CFG.grl_max_lambda,
        "domain_loss_weight":CFG.domain_loss_weight,
        "center_loss_weight":CFG.center_loss_weight,
        "weight_decay":CFG.weight_decay,
        "augmentation":"moderate training-only noise/amplitude/channel-drop/time-mask",
    },
    "variants":{"A":"CNN-Transformer","B":"CNN-Transformer+Center","C":"CNN-Transformer+Center+delayed subject GRL"},
    "selected_variant":CFG.selected_variant,
    "strict_protocols":["within_dataset_loso","cross_dataset_zero_calibration"],
    "target_statistics_used":False,
    "target_model_selection_used":False,
    "seed":CFG.seed,
}
with open(SPEC_OUTPUT,"w",encoding="utf-8") as f: json.dump(spec,f,indent=2)
print("Saved:",SPEC_OUTPUT)

assert CACHE_PATH.exists()
assert len(cache_meta_df)==9316
assert cache_meta_df["subject"].nunique()==118
assert set(cache_meta_df["harmonized_class"].unique())==set(PRIMARY_CLASSES)
print("="*78)
print("MODULE 8-V2 QA PASSED")
print("Input/cache integrity      ✓")
print("Grouped source validation ✓")
print("Target isolation          ✓")
print("Compact Transformer       ✓")
print("No LSTM                   ✓")
print("Delayed weak GRL          ✓")
print("Center loss optional      ✓")
print("Tanh CNN/embedding        ✓")
print("Final full LOSO opt-in    ✓")


## Recommended execution order

1. Run Cells 1–13.
2. Run Cell 14 for the **engineering A/B/C pilot** on BCI-IV-2a.
3. Inspect Cell 15.
4. Lock the architecture **before** the publication-facing 118-fold run.
5. Set `CFG.run_pilot_ablation=False`, `CFG.run_selected_within_full=True`, and keep `CFG.selected_variant="C"` unless you deliberately pre-register another configuration.
6. Run the selected full within-dataset LOSO.
7. Only after that, enable and implement the cross-dataset zero-calibration run.

The notebook is intentionally conservative about hyperparameter selection so that target subjects do not become an invisible validation set.